# 03 Feature engineering
This notebook will load the structured parquets from `02_preprocess_data.ipynb` and do the following:
1. Feature engineer `disruption_severity` for the `mta_alerts` dataset, as a measure of extent of disruption to general commuters per (hour, zone)
2. Combine datasets from tlc_data and aggregate for `n_trips` and `gross_fare` per (hour, zone, service_type) 
3. Use taxi zones and `latitude`, `longitude` of distinct `station_complex_id` from `mta_ridership` to determine `n_stations` per zone 
4. Use `affected` from `mta_alerts` and `latitude`, `longitude` from `mta_ridership` to determine `n_alerts` per (hour, zone)
5. Use `coco` from `meteostat_weather`, mapped to a `weather_severity` (categorical: normal -> severe) per hour

In [ ]:
# Restart spark if required
try:
    spark.stop()
except NameError:
    pass
except Exception:
    pass

from pyspark.sql import SparkSession

# Create a spark session (which will run spark jobs)
spark = (
    SparkSession.builder.appName("MAST30034 Project 1")
    .config("spark.sql.repl.eagerEval.enabled", True) 
    .config("spark.sql.parquet.cacheMetadata", "true")
    .config("spark.sql.session.timeZone", "America/New_York") # timezone set to NYC
    .getOrCreate()
)
spark.conf.set("spark.sql.shuffle.partitions", "800")
spark.conf.set("spark.sql.adaptive.enabled", "true")

spark.sparkContext.setLogLevel("ERROR") # silence warnings

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/10 21:33:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Set up and configuration

In [ ]:
# Imports
from pathlib import Path
from pyspark.sql import functions as F
import geopandas as gpd
import matplotlib.pyplot as plt

In [30]:
# Raw spatial data from `01_fetch_data.ipynb``
taxi_zones_path = "../data/raw/taxi_zones/taxi_zones.shp"

# Input directories from `02_preprocess_data.ipynb``
processed_dir = "../data/processed"

yellow_path = f"{processed_dir}/tlc_trips/yellow"
green_path = f"{processed_dir}/tlc_trips/green"
fhvhv_path = f"{processed_dir}/tlc_trips/fhvhv"
ridership_path = f"{processed_dir}/mta_ridership"
alerts_path = f"{processed_dir}/mta_alerts"
weather_path = f"{processed_dir}/meteostat_weather"

# Output directories for `03_feature_engineering.ipynb``
features_dir = "../data/processed/features"

tlc_hourly_path = f"{features_dir}/tlc_hourly"
ridership_hourly_path = f"{features_dir}/ridership_hourly"
alerts_hourly_path = f"{features_dir}/alerts_hourly"
weather_hourly_path = f"{features_dir}/weather_hourly"
final_panel_path = f"{features_dir}/final_hourly_zone_panel"

In [ ]:
# Helper function: prints shape of sdf
def spark_shape(sdf):
    return (sdf.count(), len(sdf.columns))

## Extract TLC Trip Record features
In this section, we will aggregate and extract all the TLC data that we need as features, grouped by (PULocationID, hour)
- `n_trips_taxi`: total number of trips taken by taxi (yellow, green) in the given zone (PULocationID) during the given hour
- `gross_fare_taxi`: total amount paid by passenger(s), excluding any tips, summed over all trips by taxi in the given zone (PULocationID) during the given hour
- `n_trips_fhvhv`: total number of trips taken by FHVHV (Uber, Lyft) in the given zone (PULocationID) during the given hour
- `gross_fare_fhvhv`: total amount paid by passenger(s), excluding any tips, summed over all trips by FHVHV in the given zone (PULocationID) during the given hour
- `avg_wait_time`: average wait time (pickup time - request time) for FHVHV, in the given zone (PULocationID) during the given hour (determined by request time)

In [ ]:
# Read processed TLC data
sdf_yellow = spark.read.parquet(yellow_path)
sdf_green = spark.read.parquet(green_path)
sdf_fhvhv = (
    spark.read
    .option("recursiveFileLookup", "true")
    .parquet(fhvhv_path)
)

In [22]:
# Combine yellow + green as taxi
sdf_taxi = sdf_yellow.unionByName(sdf_green)
sdf_taxi.show(5)
spark_shape(sdf_taxi)

+-------------------+-------------------+------------+------------+--------+------------+
|    pickup_datetime|   dropoff_datetime|PULocationID|DOLocationID|subtotal|service_type|
+-------------------+-------------------+------------+------------+--------+------------+
|2026-04-01 00:40:05|2026-04-01 00:52:44|         237|          68|   21.35|      yellow|
|2026-04-01 00:09:19|2026-04-01 00:21:29|         138|          75|   45.16|      yellow|
|2026-04-01 00:15:29|2026-04-01 00:34:14|         138|         112|   41.20|      yellow|
|2026-04-01 00:04:53|2026-04-01 00:11:54|         230|         234|   14.35|      yellow|
|2026-04-01 00:37:00|2026-04-01 00:45:17|          68|         107|   16.45|      yellow|
+-------------------+-------------------+------------+------------+--------+------------+
only showing top 5 rows


(47592756, 6)

In [23]:
# Aggregate taxi trips by pickup zone-hour
sdf_taxi_hourly = (
    sdf_taxi
    .withColumn("hour", F.date_trunc("hour", F.col("pickup_datetime")))
    .groupBy("hour", "PULocationID")
    .agg(
        F.count("*").alias("n_trips_taxi"),
        F.round(F.sum("subtotal"), 2).alias("gross_fare_taxi"),
    )
)
sdf_taxi_hourly.show(5)
spark_shape(sdf_taxi_hourly)

+-------------------+------------+------------+---------------+
|               hour|PULocationID|n_trips_taxi|gross_fare_taxi|
+-------------------+------------+------------+---------------+
|2026-04-01 05:00:00|         224|           3|          57.75|
|2026-04-01 14:00:00|          90|          64|        1276.57|
|2026-04-01 14:00:00|          24|          24|         444.76|
|2026-04-02 05:00:00|         151|           3|          60.00|
|2026-04-02 15:00:00|         168|           1|          12.20|
+-------------------+------------+------------+---------------+
only showing top 5 rows


(1041826, 4)

In [24]:
# Aggregate FHVHV trips by pickup zone-hour
sdf_fhvhv_hourly = (
    sdf_fhvhv
    .withColumn("hour", F.date_trunc("hour", F.col("request_datetime"))) # for FHVHV, use request_datetime to match demand time
    .withColumn(
        "wait_time",
        (F.unix_timestamp("pickup_datetime") - F.unix_timestamp("request_datetime")) # wait_time in seconds
    )
    .groupBy("hour", "PULocationID")
    .agg(
        F.count("*").alias("n_trips_fhvhv"),
        F.round(F.sum("subtotal"), 2).alias("gross_fare_fhvhv"),
        F.round(F.avg("wait_time"), 2).alias("avg_wait_time_fhvhv"),
    )
)
sdf_fhvhv_hourly.show(5)
spark_shape(sdf_fhvhv_hourly)

+-------------------+------------+-------------+----------------+-------------------+
|               hour|PULocationID|n_trips_fhvhv|gross_fare_fhvhv|avg_wait_time_fhvhv|
+-------------------+------------+-------------+----------------+-------------------+
|2025-09-15 18:00:00|         240|           21|          472.36|             291.05|
|2025-09-15 20:00:00|         225|          172|         4220.93|              234.9|
|2025-09-15 20:00:00|          66|          126|         3928.64|             275.85|
|2025-09-15 21:00:00|         145|          217|         5699.15|             216.96|
|2025-09-15 23:00:00|         178|           17|          385.19|             225.71|
+-------------------+------------+-------------+----------------+-------------------+
only showing top 5 rows


(3125328, 5)

In [79]:
# Combine the two hourly aggregates
sdf_tlc_hourly = (
    sdf_taxi_hourly
    .join(sdf_fhvhv_hourly, on=["hour", "PULocationID"], how="full")
    .fillna({
        "n_trips_taxi": 0,
        "gross_fare_taxi": 0.0,
        "n_trips_fhvhv": 0,
        "gross_fare_fhvhv": 0.0,
    })
)
sdf_tlc_hourly.show(5)
spark_shape(sdf_tlc_hourly)

+-------------------+------------+------------+---------------+-------------+----------------+-------------------+
|               hour|PULocationID|n_trips_taxi|gross_fare_taxi|n_trips_fhvhv|gross_fare_fhvhv|avg_wait_time_fhvhv|
+-------------------+------------+------------+---------------+-------------+----------------+-------------------+
|2008-12-31 15:00:00|          55|           1|         103.90|            0|             0.0|               NULL|
|2009-01-01 00:00:00|         239|           1|          14.35|            0|             0.0|               NULL|
|2024-12-31 23:00:00|           6|           0|           0.00|            1|            23.1|              111.0|
|2024-12-31 23:00:00|          15|           0|           0.00|            1|           29.99|              414.0|
|2024-12-31 23:00:00|          23|           0|           0.00|            2|           74.36|              244.0|
+-------------------+------------+------------+---------------+-------------+---

(3139662, 7)

## Create zone-station dataframe
We need a zone-station dataframe to map `mta_ridership` data with respect to `taxi_zones`. This section will produce
- `sdf_station_zones`: zone-station dataframe which maps every distinct `station_complex_id` to a `PULocationID`, also containing interpretable columns such as `station_complex` (station name), `taxi_zone` (zone name), `_borough` (from both datasets to ensure consistency), `latitude` and `longitude`

- `sdf_ridership_zoned`: `PULocationID` merged with `sdf_ridership` using the mapping provided in `sdf_station_zones` 

In [ ]:
# Read MTA ridership
sdf_ridership = spark.read.parquet(ridership_path)

# Read taxi zones
zones = gpd.read_file(taxi_zones_path)[["LocationID", "borough", "zone", "geometry"]]

In [ ]:
# Get unique station locations from ridership and metadata
stations = sdf_ridership.select(
    "station_complex_id",
    "station_complex",
    F.col("borough").alias("mta_borough"),
    "latitude",
    "longitude"
).distinct().toPandas()

spark_shape(stations)

(station_complex_id    424
 station_complex       424
 mta_borough           424
 latitude              424
 longitude             424
 dtype: int64,
 5)

In [46]:
# Turn station lat/lon into points
stations = gpd.GeoDataFrame(
    stations,
    geometry=gpd.points_from_xy(stations["longitude"], stations["latitude"]),
    crs="EPSG:4326"    # standard GPS-style coordinates
).to_crs(zones.crs)

# Spatial join station -> taxi zone, then convert back to Spark
sdf_station_zones = spark.createDataFrame(
    gpd.sjoin(stations, zones, predicate="within")
    [[
        "station_complex_id",
        "station_complex",
        "mta_borough",
        "latitude",
        "longitude",
        "LocationID",
        "borough",
        "zone",
    ]]
    .rename(columns={
        "LocationID": "PULocationID",
        "borough": "taxi_borough",
        "zone": "taxi_zone",
    })
)
sdf_station_zones.show(5)
spark_shape(sdf_station_zones)

+------------------+--------------------+-----------+---------+----------+------------+------------+------------------+
|station_complex_id|     station_complex|mta_borough| latitude| longitude|PULocationID|taxi_borough|         taxi_zone|
+------------------+--------------------+-----------+---------+----------+------------+------------+------------------+
|                72|Fort Hamilton Pkw...|   Brooklyn|40.631386| -74.00535|          26|    Brooklyn|      Borough Park|
|               404|           28 St (6)|  Manhattan| 40.74307| -73.98426|         107|   Manhattan|          Gramercy|
|               257|    Sutphin Blvd (F)|     Queens| 40.70546| -73.81071|         130|      Queens|           Jamaica|
|               367|  St Lawrence Av (6)|      Bronx| 40.83151|-73.867615|         212|       Bronx|Soundview/Bruckner|
|               151|    145 St (A,C,B,D)|  Manhattan|40.824783|-73.944214|         116|   Manhattan|  Hamilton Heights|
+------------------+--------------------

(424, 8)

In [ ]:
# Join PULocationID onto ridership
sdf_ridership_zoned = sdf_ridership.join(
    sdf_station_zones.select("station_complex_id", "PULocationID"),
    on="station_complex_id",
    how="left"
)
sdf_ridership_zoned.show(5)
spark_shape(sdf_ridership_zoned)

+------------------+-------------------+--------------------+---------+---------+---------+----------+------------+
|station_complex_id|  transit_timestamp|     station_complex|  borough|ridership| latitude| longitude|PULocationID|
+------------------+-------------------+--------------------+---------+---------+---------+----------+------------+
|               100|2025-09-01 00:00:00|      Hewes St (M,J)| Brooklyn|     65.0| 40.70687| -73.95343|         217|
|                10|2025-09-01 00:00:00|       49 St (N,R,W)|Manhattan|    406.0|  40.7599| -73.98414|         230|
|               107|2025-09-01 00:00:00|      Broad St (J,Z)|Manhattan|      1.0|40.706474|-74.011055|          87|
|               108|2025-09-01 00:00:00|Middle Village-Me...|   Queens|      5.0|40.711395|  -73.8896|         198|
|               109|2025-09-01 00:00:00|   Fresh Pond Rd (M)|   Queens|      8.0|40.706184|-73.895874|         198|
+------------------+-------------------+--------------------+---------+-

(5140101, 8)

## Extract MTA Ridership features
This section will join two key aggregated features from `mta_ridership` to the features df `sdf_tlc_hourly`: 
1. `n_stations`: the number of stations present in the `PULocationID` (taxi zone)
2. `ridership`: the number of total ridership for all stations in `PULocationID` (taxi zone) during the given `hour`

As useful columns for downstream readability, the following are also joined:
- `borough`: the name of the borough for which the `PULocationID` (taxi zone) belongs in, joined from `mta_ridership` dataset
- `taxi_zone`: the name of the taxi zone, joined from `taxi_zones` dataset

### 1. Join `n_stations`
 

In [68]:
# Zone-level station features (from all zones)
sdf_zone_features = spark.createDataFrame(
    zones[["LocationID", "borough", "zone"]]
    .rename(columns={
        "LocationID": "PULocationID",
        "borough": "borough",
        "zone": "taxi_zone",
    })
)

# Add number of subway station complexes per taxi zone to zone features
sdf_zone_features = (
    sdf_zone_features
    .join(
        sdf_station_zones
        .groupBy("PULocationID")
        .agg(F.countDistinct("station_complex_id").alias("n_stations")),
        on="PULocationID",
        how="left"
    )
    .fillna({"n_stations": 0})
)

sdf_zone_features.show(5)
spark_shape(sdf_zone_features)

+------------+-------------+--------------------+----------+
|PULocationID|      borough|           taxi_zone|n_stations|
+------------+-------------+--------------------+----------+
|           5|Staten Island|       Arden Heights|         0|
|           4|    Manhattan|       Alphabet City|         0|
|           6|Staten Island|Arrochar/Fort Wad...|         0|
|           1|          EWR|      Newark Airport|         0|
|           3|        Bronx|Allerton/Pelham G...|         2|
+------------+-------------+--------------------+----------+
only showing top 5 rows


(263, 4)

In [80]:
# Add zone features to TLC hourly data
sdf_tlc_hourly = sdf_tlc_hourly.join(
    sdf_zone_features,
    on="PULocationID",
    how="left"
)

sdf_tlc_hourly.show(5)
spark_shape(sdf_tlc_hourly)

+------------+-------------------+------------+---------------+-------------+----------------+-------------------+-------------+--------------------+----------+
|PULocationID|               hour|n_trips_taxi|gross_fare_taxi|n_trips_fhvhv|gross_fare_fhvhv|avg_wait_time_fhvhv|      borough|           taxi_zone|n_stations|
+------------+-------------------+------------+---------------+-------------+----------------+-------------------+-------------+--------------------+----------+
|           4|2024-12-31 23:00:00|           0|           0.00|            1|           30.17|              130.0|    Manhattan|       Alphabet City|         0|
|           6|2024-12-31 23:00:00|           0|           0.00|            1|            23.1|              111.0|Staten Island|Arrochar/Fort Wad...|         0|
|           7|2024-12-31 23:00:00|           0|           0.00|            7|          267.57|             288.14|       Queens|             Astoria|         5|
|          10|2024-12-31 23:00:00|

(3139662, 10)

### 2. Join `ridership` by (PULocationID, hour)

In [73]:
# Aggregate for hourly ridership in each zone
sdf_ridership_hourly = (
    sdf_ridership_zoned
    .withColumn("hour", F.date_trunc("hour", F.col("transit_timestamp")))
    .groupBy("hour", "PULocationID")
    .agg(F.sum("ridership").alias("ridership"))
)

sdf_ridership_hourly.show(5)
spark_shape(sdf_ridership_hourly)

+-------------------+------------+---------+
|               hour|PULocationID|ridership|
+-------------------+------------+---------+
|2025-09-12 09:00:00|         153|    217.0|
|2025-09-04 22:00:00|          36|   1041.0|
|2025-09-07 14:00:00|         129|   2131.0|
|2025-09-29 03:00:00|         129|    171.0|
|2025-09-02 00:00:00|         232|     57.0|
+-------------------+------------+---------+
only showing top 5 rows


(1861510, 3)

In [81]:
# Add hourly ridership in each zone to TLC hourly 
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

sdf_tlc_hourly = sdf_tlc_hourly.join(
    sdf_ridership_hourly,
    on=["hour", "PULocationID"],
    how="left"
).fillna({"ridership": 0})

sdf_tlc_hourly.show(5) 
spark_shape(sdf_tlc_hourly)

+-------------------+------------+------------+---------------+-------------+----------------+-------------------+-------------+--------------------+----------+---------+
|               hour|PULocationID|n_trips_taxi|gross_fare_taxi|n_trips_fhvhv|gross_fare_fhvhv|avg_wait_time_fhvhv|      borough|           taxi_zone|n_stations|ridership|
+-------------------+------------+------------+---------------+-------------+----------------+-------------------+-------------+--------------------+----------+---------+
|2024-12-31 23:00:00|           4|           0|           0.00|            1|           30.17|              130.0|    Manhattan|       Alphabet City|         0|      0.0|
|2024-12-31 23:00:00|           6|           0|           0.00|            1|            23.1|              111.0|Staten Island|Arrochar/Fort Wad...|         0|      0.0|
|2024-12-31 23:00:00|          11|           0|           0.00|            2|           22.98|              347.5|     Brooklyn|          Bath Be

(3139662, 11)

In [82]:
sdf_tlc_hourly.filter(F.col("n_stations") >= 10).show(10)

+-------------------+------------+------------+---------------+-------------+----------------+-------------------+--------+-------------+----------+---------+
|               hour|PULocationID|n_trips_taxi|gross_fare_taxi|n_trips_fhvhv|gross_fare_fhvhv|avg_wait_time_fhvhv| borough|    taxi_zone|n_stations|ridership|
+-------------------+------------+------------+---------------+-------------+----------------+-------------------+--------+-------------+----------+---------+
|2025-01-04 02:00:00|          76|           0|           0.00|          110|         2147.33|             301.16|Brooklyn|East New York|        10|     56.0|
|2025-01-03 03:00:00|          76|           0|           0.00|           77|         2070.83|             338.81|Brooklyn|East New York|        10|     99.0|
|2025-01-03 06:00:00|          76|           0|           0.00|          306|         6476.79|             359.37|Brooklyn|East New York|        10|   1885.0|
|2025-01-04 07:00:00|          76|           0

## Extract MTA Alerts features
This section is the truly "feature engineering" section, we need to engineer and join the following features:
- `disruption_severity`: an extracted feature from (3) which describes the severity of disruption in the given `PULocationID` during the given `hour`
- `n_alerts`: total number of alerts affecting any station in `PULocationID` , during the given `hour` 